# Laya V3.0 — Fourier-Space Feasibility Audit for Replacing Full Attention

[![Open in Google Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vtavakkoli/TinyCeNN-LM/blob/main/notebooks/Laya_V30_Fourier_Spectral_Replacement_Audit_Colab.ipynb)

**Question:** Can frequency-space mixing recover the long-range information that mean-pooled MultiscaleCeNN loses at pretrained ModernBERT **layer 15**?

**First principles:** A full FFT is information-preserving (up to rounding), and fixed frequency multiplication is equivalent to circular convolution:

\[
Y=\mathcal{F}^{-1}\!\left[D(\omega)\odot \mathcal{F}(V)\right].
\]

This supplies global mixing but **not query-conditioned retrieval**; unlike softmax attention, its token mixing coefficients are fixed across inputs. Fourier basis alone cannot circumvent that mathematical distinction. A nonlinear **input-dependent** spectral filter would be a different architecture requiring separate tests.

**What this Colab does WITHOUT training a neural network:**

1. **Parseval/FFT reconstruction:** Verify full-spectrum reconstruction and measure the teacher attention-output signal's low-frequency concentration for 16/32/64/128/256 modes.
2. **Closed-form spectral teacher fit:** Use frozen teacher value projection \(U=W_O V(X)\), fit complex frequency responses \(D(\omega)\) using **regularized least squares** from case-disjoint training samples. Evaluate shared-frequency and per-channel-frequency filters with multiple regularization and mode cutoffs.
3. **Probe-only selection:** Choose the best spectral candidate on an independent probe set; check a separate untouched **audit split** only once.
4. **Behavioral check:** Insert the chosen Fourier replacement into pretrained layer 15, measure full decision agreement, JS divergence, gold accuracy and attention/whole-model GPU latency. Other layers remain unchanged.
5. **STOP/GO decision:** Compare to previous V2.9 MultiscaleCeNN audit cosine \(\approx0.8208\), teacher agreement \(0.834\), and the required targets (cosine≥0.95, agreement≥0.95). These historical numbers are from a DIFFERENT audit sampling/run and are contextual only.

**Crucial caveats:** The ground-truth teacher-output low-pass reconstruction is an **oracle of spectral compressibility**, not a usable replacement. The complex spectral ridge filter is a content-INDEPENDENT linear sequence mixer. It may be fast or accurate, but does not reproduce arbitrary content-dependent full attention. The case splits use only the typed-decisions dataset's TRAIN split; the official test is untouched. Models selected on probe should be judged by audit without trying to optimize against audit feedback.

**Cost:** No SGD/400-step training. A moderate number of teacher forward passes to estimate Fourier filters and evaluate the chosen candidate. Requires Colab GPU.


In [ ]:
#@title 1. Install exact training dependencies
%pip -q install "transformers==4.57.6" "datasets>=2.20" "huggingface_hub>=0.25,<1" "safetensors>=0.4"
%pip -q install -U "git+https://github.com/vtavakkoli/TinyCeNN-LM.git@main"


In [ ]:
#@title 2. Imports, frozen Laya teacher and experiment configuration
from pathlib import Path
import copy, json, math, random, time
import numpy as np
import torch
from torch import nn
import torch.nn.functional as F
from datasets import load_dataset
from huggingface_hub import snapshot_download
from transformers import AutoTokenizer

from tinycenn_lm.decision_v25 import layer_kinds, MultiscaleCeNN
from tinycenn_lm.standalone_decision import (
    QTYPES, build_checkpoint_model, build_sequence,
    collate_items, normalize_question,
)
from tinycenn_lm.decision_training import require_finite

SEED = 2030
SOURCE_MODEL = "convaiinnovations/laya-typed-decisions" #@param {type:"string"}
LAYER = 15 #@param {type:"integer"}
TARGET_LAYERS = (LAYER,)
TRAIN_CASES = 200 #@param {type:"integer"}
DEV_CASES = 160 #@param {type:"integer"}
MAX_LEN = 512 #@param {type:"integer"}
BATCH = 3 #@param {type:"integer"}
STEPS = 400 #@param {type:"integer"}
REFINE_STEPS = 1200 #@param {type:"integer"}
REFINE_TOP_K = 2 #@param {type:"integer"}
TRAIN_CACHE_BATCHES = 64 #@param {type:"integer"}
PROBE_CACHE_BATCHES = 12 #@param {type:"integer"}
AUDIT_CACHE_BATCHES = 12 #@param {type:"integer"}
CHECK_EVERY = 100 #@param {type:"integer"}
OUTPUT_DIR = Path("/content/Laya_V30_Fourier_Audit")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
assert DEV_CASES >= 4, "Need nonempty probe/audit splits"
assert REFINE_TOP_K >= 1 and TRAIN_CACHE_BATCHES > 0
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if not torch.cuda.is_available():
    raise RuntimeError("Use a CUDA GPU runtime in Google Colab.")
torch.cuda.manual_seed_all(SEED)
device = torch.device("cuda")
amp_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported(including_emulation=False) else torch.float16

source_dir = Path(snapshot_download(
    SOURCE_MODEL, allow_patterns=[
        "rl_agent_config.json", "model.safetensors", "encoder/*", "tokenizer/*",
    ],
))
teacher, source_cfg = build_checkpoint_model(source_dir)
student_eval, _ = build_checkpoint_model(source_dir)
tokenizer = AutoTokenizer.from_pretrained(str(source_dir / "tokenizer"))
teacher.to(device).eval().requires_grad_(False)
student_eval.to(device).eval().requires_grad_(False)
teacher.encoder.config.reference_compile = False
student_eval.encoder.config.reference_compile = False

kinds = layer_kinds(teacher.encoder)
full_layers = [i for i, kind in enumerate(kinds) if kind == "full_attention"]
assert LAYER in full_layers, (LAYER, full_layers)
print("GPU:", torch.cuda.get_device_name(0), "AMP:", amp_dtype)
print("Source:", SOURCE_MODEL)
print("Target:", LAYER, "all full attention layers:", full_layers)
FFT_N = MAX_LEN
TRAIN_FFT_BATCHES = TRAIN_CACHE_BATCHES
CUTOFFS = (16, 32, 64, 128, 256, FFT_N//2 + 1)
RIDGES = (0.001, 0.03, 0.3)
assert FFT_N % 2 == 0
print("Spectral analysis: no SGD; FFT_N:", FFT_N, "training teacher batches:", TRAIN_FFT_BATCHES)
print("Official test used: NO")


In [ ]:
#@title 3. Leak-free train/dev data and teacher I/O helpers
def _jsonish(x):
    if isinstance(x, str):
        try:
            return json.loads(x)
        except Exception:
            return x
    return x

def _canon_label(x):
    s = str(x).strip().lower()
    try:
        v = float(s)
        if math.isfinite(v) and v.is_integer():
            return str(int(v))
    except Exception:
        pass
    return s

def _gold_metadata(q, gold_answer):
    if not isinstance(gold_answer, dict):
        raise ValueError("missing gold answer")
    probs_raw = _jsonish(gold_answer.get("probabilities", {}))
    if not isinstance(probs_raw, dict) or len(probs_raw) < 2:
        raise ValueError("gold probabilities missing or malformed")

    gold_keys = [str(k) for k in probs_raw.keys()]
    if q["t"] == "choice":
        labels = [str(k) for k in q["crit"].keys()]
    elif q["t"] == "noul":
        labels = ["false", "true"]
    else:
        labels = list(gold_keys)
        if all(_canon_label(k).lstrip("-").isdigit() for k in labels):
            labels = sorted(labels, key=lambda k: int(_canon_label(k)))

    lookup = {_canon_label(k): float(v) for k, v in probs_raw.items()}
    if len(labels) != len(gold_keys) or any(_canon_label(k) not in lookup for k in labels):
        raise ValueError("Gold labels do not match encoded option order")

    gold_probs = np.asarray([lookup[_canon_label(k)] for k in labels], dtype=np.float64)
    gold_probs = np.clip(gold_probs, 0.0, None)
    if not np.isfinite(gold_probs).all() or gold_probs.sum() <= 0:
        raise ValueError("invalid gold probability vector")
    gold_probs /= gold_probs.sum()

    gold_label = _canon_label(gold_answer.get("label", labels[int(gold_probs.argmax())]))
    canon_labels = [_canon_label(k) for k in labels]
    if gold_label not in canon_labels:
        raise ValueError(f"gold label {gold_label!r} not in {labels!r}")
    return labels, gold_probs.tolist(), canon_labels.index(gold_label)

def make_items_from_rows(rows, seed, shuffle_items):
    out = []
    head_max_len = int(source_cfg.get("head_max_len", 192))
    for row in rows:
        state = _jsonish(row["state"])
        questions = _jsonish(row["questions"])
        gold = _jsonish(row["gold"])
        case_id = str(row.get("id", ""))
        workflow = str(row.get("workflow", "unknown"))
        state_text = state if isinstance(state, str) else json.dumps(state, ensure_ascii=False)
        state_ids = tokenizer(
            state_text.replace(tokenizer.mask_token, " "),
            add_special_tokens=False,
        )["input_ids"]

        for qid, qdef in questions.items():
            q = normalize_question(qdef)
            ids, markers = build_sequence(
                tokenizer,
                state,
                q,
                max_len=min(MAX_LEN, int(source_cfg.get("max_len", MAX_LEN))),
                head_max_len=head_max_len,
                truncate_left=isinstance(state, list),
                state_ids=state_ids,
            )
            if len(markers) < 2:
                continue
            labels, gold_probs, gold_index = _gold_metadata(q, gold[qid])
            if len(labels) != len(markers):
                raise ValueError(
                    f"{case_id}/{qid}: {len(markers)} markers vs {len(labels)} gold labels"
                )
            out.append({
                "ids": ids,
                "markers": markers,
                "qtype": QTYPES[q["t"]],
                "qtype_name": q["t"],
                "qid": str(qid),
                "case_id": case_id,
                "workflow": workflow,
                "labels": labels,
                "gold_probs": gold_probs,
                "gold_index": int(gold_index),
            })
    if shuffle_items:
        random.Random(seed).shuffle(out)
    return out

rows = list(load_dataset("LocalLLaMA/typed-decisions", "all", split="train"))
rng = random.Random(SEED)
rng.shuffle(rows)
assert TRAIN_CASES + DEV_CASES <= len(rows)
train_rows = rows[:TRAIN_CASES]
dev_rows = rows[TRAIN_CASES:TRAIN_CASES + DEV_CASES]
train_items = make_items_from_rows(train_rows, SEED, True)
# Split at the CASE level before creating multiple question items per case.
probe_rows = dev_rows[:len(dev_rows) // 2]
audit_rows = dev_rows[len(dev_rows) // 2:]
probe_items = make_items_from_rows(probe_rows, SEED + 1, True)
audit_items = make_items_from_rows(audit_rows, SEED + 2, True)
dev_items = probe_items # backward-compatible alias for training helper use
assert train_items and probe_items and audit_items
assert {str(r["id"]) for r in probe_rows}.isdisjoint({str(r["id"]) for r in audit_rows})

def direct_batch(selected):
    b = collate_items(selected, int(tokenizer.pad_token_id))
    return {k: v.to(device) for k, v in b.items()}

def batch(items, offset, n=BATCH):
    selected = [items[(offset + i) % len(items)] for i in range(n)]
    return direct_batch(selected)

def _snapshot_attention_argument(value):
    """Retain the exact forward inputs for ModernBERT SDPA replay/timing."""
    if torch.is_tensor(value):
        return value.detach()
    if isinstance(value, tuple):
        return tuple(_snapshot_attention_argument(v) for v in value)
    if isinstance(value, list):
        return [_snapshot_attention_argument(v) for v in value]
    if isinstance(value, dict):
        return {k: _snapshot_attention_argument(v) for k, v in value.items()}
    return value

def _attention_output_tensor(result):
    """ModernBERT variants return either tensor or (tensor, optional weights)."""
    return result[0] if isinstance(result, (tuple, list)) else result

def attn_io(model, layer_idx, b):
    layer = model.encoder.layers[layer_idx]
    got = {}
    def pre(mod, args, kwargs):
        # The encoder passes more than x/position_embeddings/attention_mask:
        # position_ids, sliding_window_mask and optional flags are required by
        # transformers 4.57.6 SDPA. Never infer these for benchmarking.
        got["call_args"] = _snapshot_attention_argument(tuple(args))
        got["call_kwargs"] = _snapshot_attention_argument(dict(kwargs))
        got["x"] = args[0].detach()
        got["pos"] = _snapshot_attention_argument(kwargs.get("position_embeddings"))
        got["mask"] = _snapshot_attention_argument(kwargs.get("attention_mask"))
    def post(mod, args, kwargs, out):
        got["y"] = _attention_output_tensor(out).detach()
    h1 = layer.attn.register_forward_pre_hook(pre, with_kwargs=True)
    h2 = layer.attn.register_forward_hook(post, with_kwargs=True)
    try:
        with torch.no_grad(), torch.autocast(device_type="cuda", dtype=amp_dtype):
            model(**b)
    finally:
        h1.remove()
        h2.remove()
    return got

def cache_teacher_io(layer_idx, items, batches, offset=0):
    cache = []
    for i in range(batches):
        b = batch(items, offset + i * BATCH, BATCH)
        got = attn_io(teacher, layer_idx, b)
        cache.append({
            "x": got["x"],
            "pos": got["pos"],
            "mask": got["mask"],
            "call_args": got["call_args"],
            "call_kwargs": got["call_kwargs"],
            "y": got["y"],
            "valid": b["attention_mask"].bool(),
        })
    return cache

def local_metrics_from_cache(repl, cache):
    repl.eval()
    diff_sum = power_sum = cos_sum = 0.0
    tokens = 0
    with torch.no_grad():
        for entry in cache:
            with torch.autocast(device_type="cuda", dtype=amp_dtype):
                pred, _ = repl(
                    entry["x"],
                    position_embeddings=entry["pos"],
                    attention_mask=entry["mask"],
                )
            require_finite(pred, "V3.0 spectral probe")
            valid = entry["valid"]
            m = valid[:, :, None].float()
            d = pred.shape[-1]
            diff_sum += float((((pred.float() - entry["y"].float()) ** 2) * m).sum())
            power_sum += float(((entry["y"].float() ** 2) * m).sum())
            count = int(valid.sum())
            cos_sum += float(
                F.cosine_similarity(
                    pred.float()[valid],
                    entry["y"].float()[valid],
                    dim=-1,
                ).sum()
            )
            tokens += count
    nmse = diff_sum / max(power_sum, 1e-8)
    cosine = cos_sum / max(tokens, 1)
    score = nmse + 1.10 * (1.0 - cosine)
    return {"nmse": nmse, "cosine": cosine, "score": score, "tokens": tokens}

print("train/probe/audit decisions:", len(train_items), len(probe_items), len(audit_items))
print("unique train/probe/audit cases:", len(train_rows), len(probe_rows), len(audit_rows))
assert len(probe_items) >= PROBE_CACHE_BATCHES * BATCH, "Increase DEV_CASES or reduce PROBE_CACHE_BATCHES"
assert len(audit_items) >= AUDIT_CACHE_BATCHES * BATCH, "Increase DEV_CASES or reduce AUDIT_CACHE_BATCHES"
print("official test remains untouched")


## A. Why Fourier mapping may help — and where it cannot

For a length-\(N\) sequence, the DFT is an invertible linear transformation. Its orthogonality gives Parseval's identity:

\[
\|y\|_2^2=\frac{1}{N}\sum_{f=0}^{N-1}|\hat y_f|^2.
\]

If only \(K\) Fourier modes are retained, discarded spectral energy measures the best achievable *full-sequence squared reconstruction error for that fixed Fourier subspace*. We evaluate oracle low-pass teacher output first. Boundary zeros, padding and finite-length effects matter; all results use an explicit fixed \(N=512\) FFT grid.

A **fixed complex spectral response** \(\hat y_f=D_f\hat u_f\) is a circular convolution. Its receptive field is global, so it removes CeNN's local field limit and pooling-swap invariance (full-frequency filtering can distinguish those swaps). But the same convolution kernel is applied to every input and every query. Native attention instead uses input-dependent \(A(X)=\operatorname{softmax}(Q(X)K(X)^T/\sqrt d+\mathrm{mask})\). Unless the target attention operator happens to be well approximated by a fixed convolution on the task distribution, moving to Fourier space alone is insufficient.

The experiment therefore separates **(a) frequency compressibility of the teacher output**, **(b) how well a fitted translation-invariant spectral operator predicts output from pretrained values**, and **(c) downstream behavior**. They answer different questions. No result is assumed in advance.

**Phase check:** The Fourier representation must preserve the *complex phase*, not only its magnitude. A separate magnitude-only reconstruction explicitly measures the information destroyed when phase is discarded.


In [ ]:
#@title 4. Verify FFT invertibility and assess teacher-output frequency concentration (oracle)
from tinycenn_lm.laya_lab.core import _valid_tokens

orig_attention = teacher.encoder.layers[LAYER].attn
HIDDEN = int(orig_attention.config.hidden_size)
BINS = FFT_N//2 + 1
assert FFT_N >= MAX_LEN, "FFT_N must cover all sampled inputs"

@torch.inference_mode()
def pretrained_value_output(x):
    """Frozen pretrained value projection followed by original output projection."""
    B,T,D=x.shape
    qkv=orig_attention.Wqkv(x).view(B,T,3,orig_attention.num_heads,orig_attention.head_dim)
    return orig_attention.Wo(qkv[:,:,2].reshape(B,T,D))

@torch.inference_mode()
def spectral_pair(entry):
    valid=entry["valid"]
    mask=valid[:,:,None].float()
    with torch.autocast(device_type="cuda",dtype=amp_dtype):
        vbase=pretrained_value_output(entry["x"]).float()
    X=torch.fft.rfft(vbase*mask,n=FFT_N,dim=1)
    y=entry["y"].float()*mask
    Y=torch.fft.rfft(y,n=FFT_N,dim=1)
    return dict(X=X,Y=Y,y=y,valid=valid,t=entry["x"].shape[1])

def weighted_spectral_energy(spectrum):
    """Parseval weighting of rFFT: interior positive frequencies count twice."""
    e=spectrum.abs().square()
    weights=torch.ones(e.shape[1],device=e.device,dtype=e.dtype)
    if FFT_N % 2 == 0:
        weights[1:-1]=2.0
    else:
        weights[1:]=2.0
    return e*weights[None,:,None]

def cosine_nmse(pred, target, valid):
    p=pred.float()[valid]; y=target.float()[valid]
    if p.shape[0]==0: raise RuntimeError("Empty valid-token mask")
    return dict(cosine=float(F.cosine_similarity(p,y,dim=-1).mean()),
                nmse=float((p-y).square().sum()/y.square().sum().clamp_min(1e-8)))

probe_entries=cache_teacher_io(LAYER,probe_items,PROBE_CACHE_BATCHES)
probe_spectra=[spectral_pair(e) for e in probe_entries]
cutoffs=tuple(sorted(set(int(k) for k in CUTOFFS if 0<k<=BINS)))
oracle_acc={k:[] for k in cutoffs}
fft_roundtrip=[]
phase_discarded=[]
for z in probe_spectra:
    reconstructed=torch.fft.irfft(z["Y"],n=FFT_N,dim=1)[:,:z["t"]]
    full=cosine_nmse(reconstructed,z["y"],z["valid"])
    fft_roundtrip.append(full)
    # Magnitude-only FFT loses phase/position. It is NOT equivalent to full FFT.
    magnitude_only=torch.fft.irfft(z["Y"].abs().to(torch.complex64),n=FFT_N,dim=1)[:,:z["t"]]
    phase_discarded.append(cosine_nmse(magnitude_only,z["y"],z["valid"]))
    if full["nmse"]>1e-9:
        raise RuntimeError(f"Unexpected FFT round-trip error: {full}")
    energy=weighted_spectral_energy(z["Y"])
    for k in cutoffs:
        cutoff_mask=(torch.arange(BINS,device=device)<k)
        yl=torch.fft.irfft(z["Y"]*cutoff_mask[None,:,None],n=FFT_N,dim=1)[:,:z["t"]]
        metrics=cosine_nmse(yl,z["y"],z["valid"])
        metrics["retained_spectral_energy"]=float(energy[:,:k].sum()/energy.sum().clamp_min(1e-8))
        oracle_acc[k].append(metrics)

oracle_metrics={str(k):{metric:float(np.mean([item[metric] for item in oracle_acc[k]]))
    for metric in ("cosine","nmse","retained_spectral_energy")} for k in cutoffs}
print("FFT round trip: max NMSE =",max(r["nmse"] for r in fft_roundtrip))
print("Phase discarded (magnitude-only) mean cosine:",
      round(float(np.mean([r["cosine"] for r in phase_discarded])),5))
print("Oracle teacher-output low-pass (NOT a deployable replacement):")
for k,metrics in oracle_metrics.items():
    print(f"  modes={int(k):>3} cosine={metrics['cosine']:.5f} "
          f"NMSE={metrics['nmse']:.5f} spectral_energy={metrics['retained_spectral_energy']:.1%}")


In [ ]:
#@title 5. Fit complex Fourier regression analytically (zero neural-network training steps)
# Fit Y(f,c) ~= D(f,c)*X(f,c), where X is the original pretrained value-only output.
# Per-frequency complex least squares is a fixed global convolution, not attention.
# Numerator = sum conj(X)*Y. Ridge regularization guards tiny denominators.
xy=torch.zeros(BINS,HIDDEN,device=device,dtype=torch.complex64)
xx=torch.zeros(BINS,HIDDEN,device=device,dtype=torch.float32)
train_decisions=0
with torch.inference_mode():
    for i in range(TRAIN_FFT_BATCHES):
        batch_entries=cache_teacher_io(LAYER,train_items,1,offset=i*BATCH)
        z=spectral_pair(batch_entries[0])
        xy+=(z["X"].conj()*z["Y"]).sum(0)
        xx+=z["X"].abs().square().sum(0)
        train_decisions+=len(batch_entries[0]["valid"])
        del batch_entries,z
        if (i+1)%16==0 or (i+1)==TRAIN_FFT_BATCHES:
            print(f"Spectral sufficient statistics: {i+1}/{TRAIN_FFT_BATCHES} batches ({train_decisions} decisions)")

def frequency_filter(kind, ridge, cutoff):
    if kind=="shared":
        num=xy.sum(-1,keepdim=True)
        den=xx.sum(-1,keepdim=True)
        response=num/(den+float(ridge)*den.mean().clamp_min(1e-8))
        response=response.expand(BINS,HIDDEN).clone()
    elif kind=="per_channel":
        den=xx
        response=xy/(den+float(ridge)*den.mean().clamp_min(1e-8))
    else:
        raise KeyError(kind)
    # Coefficient clipping is common to all configurations, including inference.
    amp=response.abs().clamp_min(1e-8)
    response=response*(4.0/amp).clamp(max=1.0)
    response=response.clone()
    response[int(cutoff):]=0
    return response.to(torch.complex64)

def predict_spectral(z,response):
    out=torch.fft.irfft(z["X"]*response[None,:,:],n=FFT_N,dim=1)
    return out[:,:z["t"]]

def spectral_metrics(spectra,response):
    squared_error=power=cos_sum=0.0
    tokens=0
    with torch.inference_mode():
        for z in spectra:
            y=z["y"]; valid=z["valid"]
            pred=predict_spectral(z,response)
            diff=(pred-y).square()*valid[:,:,None]
            squared_error+=float(diff.sum())
            power+=float((y.square()*valid[:,:,None]).sum())
            cos_sum+=float(F.cosine_similarity(pred.float()[valid],y.float()[valid],dim=-1).sum())
            tokens+=int(valid.sum())
    return dict(cosine=cos_sum/max(tokens,1),
                nmse=squared_error/max(power,1e-8),
                tokens=tokens)

value_identity=torch.ones(BINS,HIDDEN,device=device,dtype=torch.complex64)
baseline=spectral_metrics(probe_spectra,value_identity)
print("Frozen native value-only baseline: cosine=",round(baseline["cosine"],5),
      "NMSE=",round(baseline["nmse"],5))
screen=[]
for kind in ("shared","per_channel"):
    for ridge in RIDGES:
        for cutoff in cutoffs:
            resp=frequency_filter(kind,ridge,cutoff)
            quality=spectral_metrics(probe_spectra,resp)
            record=dict(kind=kind,ridge=float(ridge),cutoff=int(cutoff),
                        probe=quality,selection_score=quality["nmse"]+1.1*(1-quality["cosine"]))
            screen.append(record)
            print(f"{kind:<11} ridge={ridge:.3g} modes={cutoff:>3} "
                  f"probe cos={quality['cosine']:.5f} nmse={quality['nmse']:.5f}")
screen.sort(key=lambda m:(m["selection_score"],-m["probe"]["cosine"]))
winner=screen[0]
best_response=frequency_filter(winner["kind"],winner["ridge"],winner["cutoff"])
print("\nPROBE-SELECTED spectral candidate (audit NOT used for selection):",winner)


In [ ]:
#@title 6. Once-only held-out audit and layer 15 replacement compatibility
# We don't revisit the audit to choose another spectral candidate.
audit_entries=cache_teacher_io(LAYER,audit_items,AUDIT_CACHE_BATCHES)
audit_spectra=[spectral_pair(e) for e in audit_entries]
audit_metrics=spectral_metrics(audit_spectra,best_response)
print("HELD-OUT AUDIT: cosine={:.5f} NMSE={:.5f}".format(
    audit_metrics["cosine"],audit_metrics["nmse"]))

class FourierSpectralReplacement(nn.Module):
    """Deployable fixed frequency-domain layer. O(N log N), no QK/SDPA.

    Input length must be <=FFT_N. Uses FFT_N zero-padded circular convolution
    and keeps only the original length; padding is masked at input and output.
    """
    architecture="v30_complex_fourier_ridge"

    def __init__(self,original,response,fft_n):
        super().__init__()
        self.hidden_size=int(original.config.hidden_size)
        self.num_heads=int(original.num_heads)
        self.head_dim=int(original.head_dim)
        self.fft_n=int(fft_n)
        self.Wqkv=copy.deepcopy(original.Wqkv).requires_grad_(False)
        self.Wo=copy.deepcopy(original.Wo).requires_grad_(False)
        self.register_buffer("response",response.detach().clone().to(torch.complex64))

    def forward(self,hidden_states,attention_mask=None,**kwargs):
        B,T,D=hidden_states.shape
        if T>self.fft_n:
            raise ValueError(f"Sequence T={T} exceeds trained fixed FFT_N={self.fft_n}")
        valid=_valid_tokens(attention_mask,hidden_states)
        qkv=self.Wqkv(hidden_states).view(B,T,3,self.num_heads,self.head_dim)
        u=self.Wo(qkv[:,:,2].reshape(B,T,D))
        mask=valid[:,:,None].float()
        xf=torch.fft.rfft(u.float()*mask,n=self.fft_n,dim=1)
        yf=xf*self.response[None,:,:]
        result=torch.fft.irfft(yf,n=self.fft_n,dim=1)[:,:T]*mask
        return result.to(hidden_states.dtype),None

replacement=FourierSpectralReplacement(orig_attention,best_response,FFT_N).to(device).eval()
# Check that the same spectral math is executed by the real ModernBERT replacement.
with torch.inference_mode(),torch.autocast(device_type="cuda",dtype=amp_dtype):
    direct,_=replacement(audit_entries[0]["x"],attention_mask=audit_entries[0]["mask"])
first=predict_spectral(audit_spectra[0],best_response)
match=cosine_nmse(direct,first,audit_entries[0]["valid"])
assert match["nmse"] < 1e-5, f"Spectral wrapper differs from fitting code: {match}"
print("Replacement mathematical consistency PASS:", match)


In [ ]:
#@title 7. End-to-end decision agreement/JS and fair GPU timing
@torch.inference_mode()
def validate_student(student,items):
    n=matched=teacher_correct=student_correct=0
    js=[]
    for offset in range(0,len(items),BATCH):
        chosen=items[offset:offset+BATCH]
        b=direct_batch(chosen)
        with torch.autocast(device_type="cuda",dtype=amp_dtype):
            tl,_=teacher(**b); sl,_=student(**b)
        allowed=b["marker_mask"].bool()
        tp=torch.softmax(tl.float().masked_fill(~allowed,-1e9),dim=-1)
        sp=torch.softmax(sl.float().masked_fill(~allowed,-1e9),dim=-1)
        gold=torch.tensor([item["gold_index"] for item in chosen],device=device)
        n+=len(chosen)
        matched+=int((tp.argmax(-1)==sp.argmax(-1)).sum())
        teacher_correct+=int((tp.argmax(-1)==gold).sum())
        student_correct+=int((sp.argmax(-1)==gold).sum())
        mid=.5*(tp+sp)
        div=.5*((tp*(tp.clamp_min(1e-8).log()-mid.clamp_min(1e-8).log())).sum(-1))
        div+=.5*((sp*(sp.clamp_min(1e-8).log()-mid.clamp_min(1e-8).log())).sum(-1))
        js.extend(div.detach().cpu().tolist())
    return dict(decisions=n,agreement=matched/max(n,1),mean_js=float(np.mean(js)),
                teacher_gold_accuracy=teacher_correct/max(n,1),
                student_gold_accuracy=student_correct/max(n,1),
                gold_accuracy_drop=(teacher_correct-student_correct)/max(n,1))

@torch.inference_mode()
def measured_ms(fn,warmup=6,repeats=20):
    with torch.autocast(device_type="cuda",dtype=amp_dtype):
        for _ in range(warmup):
            _=fn()
        torch.cuda.synchronize()
        times=[]
        for _ in range(repeats):
            tick=time.perf_counter()
            _=fn()
            torch.cuda.synchronize()
            times.append((time.perf_counter()-tick)*1000)
    return float(np.median(times))

original=student_eval.encoder.layers[LAYER].attn
entry=audit_entries[0]
fixed=direct_batch(audit_items[:BATCH])
try:
    student_eval.encoder.layers[LAYER].attn=replacement
    student_eval.eval()
    quality=validate_student(student_eval,audit_items)
    teacher_layer_ms=measured_ms(lambda:orig_attention(*entry["call_args"],**entry["call_kwargs"]))
    spectral_layer_ms=measured_ms(lambda:replacement(entry["x"],attention_mask=entry["mask"]))
    teacher_model_ms=measured_ms(lambda:teacher(**fixed),warmup=3,repeats=10)
    spectral_model_ms=measured_ms(lambda:student_eval(**fixed),warmup=3,repeats=10)
finally:
    student_eval.encoder.layers[LAYER].attn=original
    student_eval.eval().requires_grad_(False)

speed=dict(teacher_attention_ms=teacher_layer_ms,
           spectral_attention_ms=spectral_layer_ms,
           layer_speedup=teacher_layer_ms/max(spectral_layer_ms,1e-8),
           teacher_model_ms=teacher_model_ms,
           spectral_model_ms=spectral_model_ms,
           whole_model_speedup=teacher_model_ms/max(spectral_model_ms,1e-8))
print("\nLayer 15 Fourier model held-out decision results:",quality)
print("Measured latency:",speed)
print("Caution: one layer replaced; whole-model latency dominates at small sequence lengths.")


In [ ]:
#@title 8. Mathematical decision, report export, visual diagnostics
target_cos=.95
target_agreement=.95
ready=(audit_metrics["cosine"]>=target_cos and quality["agreement"]>=target_agreement
       and speed["layer_speedup"]>1.0)
print("\n" + "="*92)
print("V3.0 FFT-SPECTRAL MATHEMATICAL FEASIBILITY REPORT")
print("="*92)
print(f"Teacher-output lowpass K=32:  cosine={oracle_metrics['32']['cosine']:.5f}" if "32" in oracle_metrics else "K32 unavailable")
print(f"Frozen VALUE baseline: cosine={baseline['cosine']:.5f}")
print(f"Best ridge spectral model AUDIT: cosine={audit_metrics['cosine']:.5f}, "
      f"NMSE={audit_metrics['nmse']:.5f}, teacher agreement={quality['agreement']:.4f}, "
      f"layer speedup={speed['layer_speedup']:.3f}x")
print("Meets predeclared fidelity+agreement+speed criteria:",ready)
if not ready:
    print("NOT READY: frequency-space mapping/linear filtering alone is insufficient under these criteria.")
    print("Next candidate: input-dependent spectral filters or selective key-value memory, "
          "with a separate mathematical test before 400-step distillation.")
else:
    print("PROMISING in this one-layer experiment; repeat across different seeds, "
          "sequence lengths and tasks before concluding equivalence.")
report=dict(
    version="3.0-fourier-math-audit",model=SOURCE_MODEL,layer=LAYER,
    neural_training_updates=0,fft_length=FFT_N,official_test_used=False,
    train_cases=TRAIN_CASES,probe_cases=len(probe_rows),audit_cases=len(audit_rows),
    analytic_train_decisions=train_decisions,
    fft_roundtrip=fft_roundtrip,phase_discarded=phase_discarded,
    oracle_teacher_output_lowpass=oracle_metrics,
    value_only_probe=baseline,
    probe_winner=winner,
    selected_audit_local=audit_metrics,selected_audit_e2e=quality,
    measured_latency=speed,
    pass_target=bool(ready),
    spectral_candidate_results=screen,
    caveats=[
        "Teacher output low-pass is an oracle compression check, not a replacement",
        "Fixed Fourier multiplier is content-independent circular convolution",
        "Complex frequency response was fit by ridge least squares, no SGD",
        "The holdout audit was never used for hyperparameter choice",
        "Historical V2.9 CeNN results have different sampled examples",
        "Only one layer was replaced; official test untouched",
        "FFT resolution fixed to 512 with zero padding and masked output",
    ],
)
report_path=OUTPUT_DIR/"v30_fourier_mathematical_audit.json"
report_path.write_text(json.dumps(report,indent=2,allow_nan=False),encoding="utf-8")
torch.save(dict(
    source_model=SOURCE_MODEL,layer=LAYER,fft_n=FFT_N,
    configuration={k:winner[k] for k in ("kind","ridge","cutoff")},
    response=best_response.detach().cpu(),
),OUTPUT_DIR/"v30_best_frequency_filter.pt")
print("Saved report:",report_path)

import matplotlib.pyplot as plt
ks=[int(k) for k in oracle_metrics]
vals=[oracle_metrics[str(k)]["cosine"] for k in ks]
plt.plot(ks,vals,marker="o",label="Oracle: true teacher output low-pass")
plt.axhline(target_cos,linestyle="--",label="0.95 target")
plt.axhline(audit_metrics["cosine"],linestyle=":",label="Spectral model held-out audit")
plt.xlabel("Retained FFT bins (out of 257)")
plt.ylabel("Output cosine")
plt.title("Frequency compressibility vs. predictive replacement")
plt.legend()
plt.grid(alpha=.3)
plt.show()
